# 🌊 Flood Depth Model — Water-Aware GPU Training
## Option 3: Water-Aware Fine-Tuning (2-3 Hours, Best Accuracy Boost)

This notebook trains your flood depth model to focus **only on water pixels**,
so it learns accurate depth from partially-flooded images (one side water, one side dry).

---
### ✅ Before you start — read this!
1. **Enable GPU first** — see Step 1 below.
2. **Run every cell in order** — do not skip any cell.
3. **Keep the browser tab open** — closing it stops training.
4. **Your images must be named with depth** — e.g. `flood_depth25cm.jpg` (depth in cm in the filename).
5. **Minimum data**: 20+ training images, 3+ validation images.

**Cost:** $0 (free Colab GPU)  
**Time:** ~2-3 hours training + ~10 min setup  
**Result:** `models/best_flood_model_water_aware.pth` saved to Google Drive

---
## STEP 1 — Enable GPU ⚠️ DO THIS FIRST!

1. Click **Runtime** in the top menu
2. Click **Change runtime type**
3. Under **Hardware accelerator**, select **GPU** (T4 GPU is free)
4. Click **Save**
5. The page will reload — come back and run Step 2 onwards

Then run the cell below to confirm GPU is active:

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU ready: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
else:
    print("❌ GPU not found!")
    print("   → Go to Runtime → Change runtime type → Select GPU → Save")
    raise SystemExit("Enable GPU before continuing.")

---
## STEP 2 — Mount Google Drive

This lets us save the trained model to your Drive so it persists after Colab disconnects.

When you run this cell:
- A link will appear — click it
- Sign in with your Google account
- Copy the code that appears and paste it into the box
- Press Enter

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted at /content/drive")
!ls /content/drive/MyDrive | head -10

---
## STEP 3 — Clone the Repository

This downloads your flood-depth-estimator code into `/content/flood-depth-estimator`.

⚠️ **Important:** We clone the `kaumod-configure-git-lfs` branch — this branch contains
the water-aware training code. The `main` branch does NOT have this yet.

If you already ran this before in this session, no need to re-run — it will skip.

In [ ]:
import os

REPO_DIR = '/content/flood-depth-estimator'

if os.path.exists(REPO_DIR):
    print("✅ Repo already exists — pulling latest changes")
    !git -C {REPO_DIR} pull origin kaumod-configure-git-lfs
else:
    print("📥 Cloning repository (branch: kaumod-configure-git-lfs)...")
    !git clone --branch kaumod-configure-git-lfs --single-branch \
        https://github.com/Mishra-Kaumod/flood-depth-estimator.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"\n✅ Working directory: {os.getcwd()}")

# Confirm critical files exist
critical = ['src/train_water_aware.py', 'src/dataset.py', 'src/train.py',
            'src/water_region_detector.py', 'config/config.yaml']
for f in critical:
    status = '✅' if os.path.exists(f) else '❌ MISSING'
    print(f"  {status}  {f}")

---
## STEP 4 — Install Required Packages

Colab already has PyTorch, but we need a few extra packages.
This takes about 1-2 minutes.

In [ ]:
print("📦 Installing required packages...")
!pip install -q tqdm pyyaml opencv-python-headless
print("✅ Packages ready")

# Set num_workers to 2 (Colab recommendation to avoid DataLoader freeze warnings)
import subprocess
result = subprocess.run(['sed', '-i', 's/num_workers: 4/num_workers: 2/',
                        'config/config.yaml'], capture_output=True)
print("✅ DataLoader workers set to 2 (optimal for Colab)")

---
## STEP 5 — Upload Your Training Images

**⚠️ IMPORTANT — Image naming format:**
Your filenames **must** contain the water depth in centimetres, like:
- `road_flood_depth25cm.jpg` → 25 cm water depth
- `street_depth50cm.png` → 50 cm water depth
- `flood_depth10cm.jpeg` → 10 cm water depth

If your images do NOT follow this naming, they will be treated as 0 cm depth (incorrect).

**How many images do I need?**
- Minimum: 20 training + 3 validation
- Recommended: 100+ training + 20+ validation
- More = better accuracy

Run the cell below, then click **Choose Files** to upload your TRAINING images.

In [ ]:
from google.colab import files
import os, shutil

# Create directory structure
os.makedirs('flood_dataset/train', exist_ok=True)
os.makedirs('flood_dataset/val', exist_ok=True)

print("📂 Upload your TRAINING images now...")
print("   (filenames must contain depthXXcm e.g. flood_depth25cm.jpg)")
print()

uploaded_train = files.upload()

moved = []
for fname in uploaded_train.keys():
    dest = f'flood_dataset/train/{fname}'
    if os.path.exists(fname):
        shutil.move(fname, dest)
    else:
        # Colab sometimes writes to /content directly
        src = f'/content/{fname}'
        if os.path.exists(src):
            shutil.move(src, dest)
    moved.append(dest)

print(f"\n✅ Moved {len(moved)} training images to flood_dataset/train/")
print(f"   Total training images: {len(os.listdir('flood_dataset/train'))}")

---
## STEP 6 — Upload Validation Images

Validation images are used to measure how well the model is improving during training.
Use images the model has NOT seen in training — ideally from different locations/conditions.

Same naming rule: filenames must contain `depthXXcm`.

**If you don't have separate validation images:** Run the cell below and it will
automatically split 20% of your training images into validation.

In [ ]:
from google.colab import files
import os, shutil, random

# Option A: Upload separate validation images
# ─────────────────────────────────────────────
# Uncomment the block below if you have separate validation images:
#
# print("📂 Upload your VALIDATION images now...")
# uploaded_val = files.upload()
# for fname in uploaded_val.keys():
#     dest = f'flood_dataset/val/{fname}'
#     src = fname if os.path.exists(fname) else f'/content/{fname}'
#     shutil.move(src, dest)
# print(f"✅ {len(uploaded_val)} validation images uploaded")


# Option B: Auto-split 20% from training images (default)
# ─────────────────────────────────────────────────────────
train_imgs = os.listdir('flood_dataset/train')
val_imgs   = os.listdir('flood_dataset/val')

if len(val_imgs) == 0:
    print("ℹ️  No validation images found — auto-splitting 20% from training set")
    random.shuffle(train_imgs)
    n_val = max(3, len(train_imgs) // 5)          # at least 3 val images
    for fname in train_imgs[:n_val]:
        shutil.move(f'flood_dataset/train/{fname}',
                    f'flood_dataset/val/{fname}')
    print(f"   Moved {n_val} images → flood_dataset/val/")

train_count = len(os.listdir('flood_dataset/train'))
val_count   = len(os.listdir('flood_dataset/val'))
print(f"\n✅ Dataset ready:")
print(f"   Training images  : {train_count}")
print(f"   Validation images: {val_count}")

if train_count == 0:
    raise ValueError("❌ No training images found! Run Step 5 first.")
if val_count == 0:
    raise ValueError("❌ No validation images found! Upload some or set n_val > 0.")

---
## STEP 7 — Label Your Images (CRITICAL — Read This!)

### Why this step matters
The model learns by comparing its prediction to the **actual depth label** for each image.
If labels are wrong or missing, the model learns to predict **0 cm for everything** (label collapse).

### You have TWO options:

**Option A — Rename your images (simplest):**
Include the depth in centimetres in the filename:
```
flood_road_depth25cm.jpg    → 25 cm depth
street_flooded_depth50cm.png → 50 cm depth
dry_road_depth0cm.jpg       → 0 cm (no flood)
```

**Option B — Create a labels.csv file (easier if you have many images):**
Run the cell below to auto-generate a `labels.csv` you can edit.

⚠️ **If all your images show 0 labeled below, STOP and fix labels before training!**

In [ ]:
import os, re, csv
from pathlib import Path

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

def analyze_and_label(split):
    folder = Path(f'flood_dataset/{split}')
    imgs = sorted([f for f in folder.rglob('*') if f.suffix.lower() in IMG_EXTS])

    # Check for existing labels.csv
    csv_path = folder / 'labels.csv'
    existing = {}
    if csv_path.exists():
        with open(csv_path) as f:
            for row in csv.DictReader(f):
                try: existing[row['filename']] = int(float(row['depth_cm']))
                except: pass

    labeled, unlabeled = [], []
    rows = []
    for img in imgs:
        if img.name in existing:
            depth = existing[img.name]
        else:
            m = re.search(r'depth(\d+)cm', img.name.lower())
            depth = int(m.group(1)) if m else None
        rows.append({'filename': img.name, 'depth_cm': depth})
        (labeled if depth is not None else unlabeled).append(img.name)

    print(f'\n{"="*60}')
    print(f'  {split.upper()} SET: {len(imgs)} images')
    print(f'  Labeled  : {len(labeled)} images')
    print(f'  UNLABELED: {len(unlabeled)} images  <- depth defaults to 0!')
    if labeled:
        depths = [r["depth_cm"] for r in rows if r["depth_cm"] is not None]
        zeros  = depths.count(0)
        print(f'  Depth range: {min(depths)}-{max(depths)} cm  ({zeros} are 0 cm)')
        if zeros / len(depths) > 0.8:
            print()
            print('  *** LABEL COLLAPSE RISK ***')
            print('  Over 80% of labels are 0 cm!')
            print('  The model will learn to predict 0 for everything.')
            print('  Please assign real depth values before training!')
    print(f'  {"="*60}')

    # Show sample
    print(f'  Sample labels (first 10):')
    for r in rows[:10]:
        status = '✅' if r['depth_cm'] is not None and r['depth_cm'] > 0 else '⚠️ '
        print(f'    {status}  {r["filename"][:50]:50s}  {r["depth_cm"]} cm')
    if len(rows) > 10:
        print(f'    ... and {len(rows)-10} more')

    return rows, folder

train_rows, train_dir = analyze_and_label('train')
val_rows, val_dir     = analyze_and_label('val')


### If images are unlabeled — assign depths now

Edit the dictionary below with `filename: depth_cm` for any unlabeled images.
Then run the next cell to write `labels.csv`.

Rough depth guide:
- Ankle-deep water visible: **10-20 cm**
- Knee-deep flooding: **30-50 cm**
- Waist-deep / vehicle submerged: **60-90 cm**
- Extreme flooding: **90-150 cm**

In [ ]:
import csv

# ─────────────────────────────────────────────────────────────
# EDIT THIS DICTIONARY: add 'filename': depth_cm for each image
# that was shown as '⚠️  None' in the cell above.
# Leave empty {} if all images already have labels from filenames.
# ─────────────────────────────────────────────────────────────
MANUAL_LABELS = {
    # 'flood_image1.jpg': 30,
    # 'IMG_4521.jpg':     50,
    # 'street_water.png': 20,
}

def save_labels_csv(rows, folder, manual):
    final = []
    for r in rows:
        depth = manual.get(r['filename'], r['depth_cm'])
        if depth is None:
            depth = 0  # last resort default — will trigger warning
        final.append({'filename': r['filename'], 'depth_cm': depth})

    csv_path = folder / 'labels.csv'
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['filename', 'depth_cm'])
        writer.writeheader()
        writer.writerows(final)

    depths = [r['depth_cm'] for r in final]
    zeros  = depths.count(0)
    print(f'Saved {len(final)} labels to {csv_path}')
    print(f'  Depth range: {min(depths)}-{max(depths)} cm  ({zeros} zeros)')
    if zeros / len(depths) > 0.8:
        print()
        print('  ⚠️  STILL OVER 80% ZEROS — model will collapse!')
        print('  Please fill in MANUAL_LABELS dict above and re-run this cell.')
    else:
        print('  ✅ Label distribution looks good — safe to train!')
    return final

print('=== TRAINING LABELS ===')
save_labels_csv(train_rows, train_dir, MANUAL_LABELS)
print('\n=== VALIDATION LABELS ===')
save_labels_csv(val_rows, val_dir, MANUAL_LABELS)


---
## STEP 8 — 💎 Run Water-Aware Training (GPU)

This is the main training step. Here is exactly what happens:
1. Loads EfficientNet-B0 with ImageNet weights
2. For every image batch, detects water regions using HSV + RGB + contrast analysis
3. Computes loss **weighted by water coverage** — images with more water get higher weight
4. Saves the best model every epoch to `models/best_flood_model_water_aware.pth`
5. Uses early stopping — training stops automatically if the model stops improving

**Expected duration:** 2-3 hours for 20 epochs with 400+ training images on a T4 GPU.

You will see progress like:
```
EPOCH 1/20  →  Train Loss: 0.045 | Val Loss: 0.038
EPOCH 2/20  →  Train Loss: 0.038 | Val Loss: 0.031
...
✅ Best water-aware model saved to models/best_flood_model_water_aware.pth
```

⚠️ **Keep this browser tab open.** Closing it stops training.

⚠️ **Training loss starting near 0 (< 0.001) = label collapse** — stop and fix labels!

In [ ]:
import os
os.chdir('/content/flood-depth-estimator')

assert os.path.exists('src/train_water_aware.py'), \
    '❌ src/train_water_aware.py not found! Re-run Step 3.'
assert os.path.exists('flood_dataset/train'), \
    '❌ flood_dataset/train not found! Re-run Steps 5-6.'
assert len(os.listdir('flood_dataset/train')) > 0, \
    '❌ No training images! Re-run Steps 5-6.'

# Check labels.csv exists or filenames have depth info
import re
from pathlib import Path
train_imgs = list(Path('flood_dataset/train').glob('*.jpg')) + \
             list(Path('flood_dataset/train').glob('*.png'))
has_csv = Path('flood_dataset/train/labels.csv').exists()
has_names = sum(1 for f in train_imgs if re.search(r'depth\d+cm', f.name.lower()))

if not has_csv and has_names == 0:
    raise ValueError(
        '❌ LABEL COLLAPSE RISK: No labels.csv found AND no filenames contain depthXXcm.\n'
        '   The model will train on all-zero labels and always predict 0 cm.\n'
        '   Please complete Step 7 (create labels.csv) before continuing.'
    )
elif not has_csv and has_names < len(train_imgs) * 0.5:
    print(f'⚠️  Only {has_names}/{len(train_imgs)} images have depth in filename.')
    print('   Consider running Step 7 to create labels.csv for the rest.')
else:
    print('✅ All checks passed — labels look OK')

print('\n✅ Starting water-aware training...')
print('   Watch for Train Loss starting around 0.01-0.10 (good)')
print('   If Train Loss starts near 0.000001 → STOP — label collapse!')
print()

!python -m src.train_water_aware --config config/config.yaml


---
## STEP 9 — Check Training Results

Run this after training finishes to see what models were saved.

In [ ]:
import os
os.chdir('/content/flood-depth-estimator')

print("📦 Models saved:")
if os.path.exists('models'):
    for f in sorted(os.listdir('models')):
        fpath = f'models/{f}'
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"   ✅ {f}  ({size_mb:.1f} MB)")
else:
    print("   ❌ No models folder found — did training complete?")

best = 'models/best_flood_model_water_aware.pth'
if os.path.exists(best):
    import torch
    ckpt = torch.load(best, map_location='cpu')
    print(f"\n🏆 Best model info:")
    print(f"   Epoch   : {ckpt.get('epoch', 'N/A')}")
    print(f"   Val Loss: {ckpt.get('best_val_loss', 'N/A'):.6f}")
    hist = ckpt.get('training_history', {})
    if hist.get('train_loss'):
        print(f"   Train loss history: {[round(x,4) for x in hist['train_loss']]}")
    if hist.get('val_loss'):
        print(f"   Val loss history  : {[round(x,4) for x in hist['val_loss']]}")

---
## STEP 10 — Save Model to Google Drive

Save the trained model to Google Drive so it persists even after this Colab session ends.
You can then download it from Drive at any time.

In [ ]:
import os, shutil
os.chdir('/content/flood-depth-estimator')

drive_dest = '/content/drive/MyDrive/flood_depth_models'
os.makedirs(drive_dest, exist_ok=True)

saved = []
if os.path.exists('models'):
    for f in os.listdir('models'):
        if f.endswith('.pth'):
            src = f'models/{f}'
            dst = f'{drive_dest}/{f}'
            shutil.copy(src, dst)
            saved.append(f)

if saved:
    print(f"✅ Saved {len(saved)} model(s) to Google Drive:")
    for f in saved:
        print(f"   → {drive_dest}/{f}")
    print("\n   Access them at: My Drive → flood_depth_models")
else:
    print("❌ No .pth files found in models/ — did training complete?")

---
## STEP 11 — Download Model to Your Computer

This downloads the best trained model directly to your computer.
Use this model in your local `models/` folder to replace the old one.

In [ ]:
from google.colab import files
import os
os.chdir('/content/flood-depth-estimator')

best_model = 'models/best_flood_model_water_aware.pth'

if os.path.exists(best_model):
    print(f"📥 Downloading {best_model} to your computer...")
    files.download(best_model)
    print("✅ Download started!")
    print()
    print("📋 Next steps after download:")
    print("   1. Copy the file to your local repo:  models/best_flood_model_water_aware.pth")
    print("   2. Update config/config.yaml:  best_model_path: models/best_flood_model_water_aware.pth")
    print("   3. Or rename it to:  models/best_flood_model.pth  to use as default")
else:
    print(f"❌ {best_model} not found!")
    print("\nAvailable models:")
    if os.path.exists('models'):
        for f in os.listdir('models'):
            print(f"  → models/{f}")
    else:
        print("  No models directory — did training complete?")

---
## 🚨 Troubleshooting

| Error | Fix |
|-------|-----|
| `GPU not found` | Runtime → Change runtime type → GPU → Save |
| `No module named 'src'` | Re-run Step 3 (clone), then re-run Step 8 |
| `Dataset directory not found` | Re-run Steps 5-6 to upload images |
| `num_samples = 0` | Upload images first in Step 5, then re-run Step 8 |
| `CUDA out of memory` | Lower batch size: `!sed -i 's/batch_size: 32/batch_size: 16/' config/config.yaml` |
| `Session disconnected` | Models saved to Drive in Step 10 — reconnect and continue |
| Training loss stuck at same value | Images have no `depthXXcm` in filename — rename them |

---
## 💡 Tips

- **Resume after disconnect:** Colab sessions expire after ~12h. Always save to Drive (Step 10) at the end.
- **Reduce epochs for quick test:** `!sed -i 's/epochs: 20/epochs: 3/' config/config.yaml` before Step 8
- **Check GPU usage:** `!nvidia-smi` in a new cell during training
- **Rename images without depth:** Use pattern like `img01_depth15cm.jpg` for 15cm water depth